In [1]:
import math
import pandas as pd
import numpy as np
import time
from collections import defaultdict
from gurobipy import Model, GRB, quicksum
import opendssdirect as dss 
dss.Text.Command(r'Redirect "C:\Users\lfang7\Project\Distribution network\OpenDSS Model\Master.dss"')
import random
seed = 42 
np.random.seed(seed)  
random.seed(seed) 

In [2]:
baseMVA = 10
basekV = 6.9
baseI = baseMVA * 1000 / (math.sqrt(3) * basekV)
I_max = 1000 * 1000 / baseI**2
Zb = basekV**2 / baseMVA
V_max = 1.05**2  
V_min = 0.95**2  

In [3]:
I_max

1.4283000000000001

In [4]:
# Load Active Power from Excel files into NumPy arrays using pandas.
feederA_P = pd.read_excel('Calculated Nodal P&Q.xlsx', sheet_name='FeederA_P')  # Load active power data for Feeder A
feederB_P = pd.read_excel('Calculated Nodal P&Q.xlsx', sheet_name='FeederB_P')  # Load active power data for Feeder B
feederC_P = pd.read_excel('Calculated Nodal P&Q.xlsx', sheet_name='FeederC_P')  # Load active power data for Feeder C

# Load Reactive Power from Excel files into NumPy arrays using pandas.
feederA_Q = pd.read_excel('Calculated Nodal P&Q.xlsx', sheet_name='FeederA_Q')  # Load reactive power data for Feeder A
feederB_Q = pd.read_excel('Calculated Nodal P&Q.xlsx', sheet_name='FeederB_Q')  # Load reactive power data for Feeder B
feederC_Q = pd.read_excel('Calculated Nodal P&Q.xlsx', sheet_name='FeederC_Q')  # Load reactive power data for Feeder C

columnsA_P = feederA_P.columns.tolist()   
columnsB_P = feederB_P.columns.tolist()
columnsC_P = feederC_P.columns.tolist()
columnsA_Q = feederA_Q.columns.tolist()
columnsB_Q = feederB_Q.columns.tolist()
columnsC_Q = feederC_Q.columns.tolist()

# Rename the first column
columnsA_P[0] = 'Time'
columnsB_P[0] = 'Time'
columnsC_P[0] = 'Time'
columnsA_Q[0] = 'Time'
columnsB_Q[0] = 'Time'
columnsC_Q[0] = 'Time'

# Assign the updated column names back to the DataFrame
feederA_P.columns = columnsA_P
feederB_P.columns = columnsB_P
feederC_P.columns = columnsC_P
feederA_Q.columns = columnsA_Q
feederB_Q.columns = columnsB_Q
feederC_Q.columns = columnsC_Q

feederA_P['Time'] = pd.to_datetime(feederA_P['Time'])
feederB_P['Time'] = pd.to_datetime(feederB_P['Time'])
feederC_P['Time'] = pd.to_datetime(feederC_P['Time'])
feederA_Q['Time'] = pd.to_datetime(feederA_Q['Time'])
feederB_Q['Time'] = pd.to_datetime(feederB_Q['Time'])
feederC_Q['Time'] = pd.to_datetime(feederC_Q['Time'])

feederA_P.set_index(feederA_P.columns[0], inplace=True)
feederB_P.set_index(feederB_P.columns[0], inplace=True)
feederC_P.set_index(feederC_P.columns[0], inplace=True)
feederA_Q.set_index(feederA_Q.columns[0], inplace=True)
feederB_Q.set_index(feederB_Q.columns[0], inplace=True)
feederC_Q.set_index(feederC_Q.columns[0], inplace=True)

feeders_P = pd.concat([feederA_P, feederB_P, feederC_P], axis=1)
feeders_Q = pd.concat([feederA_Q, feederB_Q, feederC_Q], axis=1)

In [5]:
feeders_P

,Bus 1001,Bus 1002,Bus 1003,Bus 1004,Bus 1005,Bus 1006,Bus 1007,Bus 1008,Bus 1009,Bus 1010,...,Bus 3153,Bus 3154,Bus 3155,Bus 3156,Bus 3157,Bus 3158,Bus 3159,Bus 3160,Bus 3161,Bus 3162
Time,,,,,,,,,,,,,,,,,,,,,
2017-01-01 01:00:00.000,0,0,15.290,6.892,4.916,5.04,4.163,14.0960,17.081,7.136,...,5.9760,4.836,1.47700,0,9.740,6.517,9.776,4.252,2.744,3.750
2017-01-01 02:00:00.000,0,0,14.901,6.672,5.335,4.76,3.070,14.9370,12.786,7.078,...,7.1270,6.638,1.48800,0,7.721,6.858,9.565,4.154,3.468,3.710
2017-01-01 03:00:00.005,0,0,15.772,7.013,4.563,5.04,3.507,14.7890,10.209,5.991,...,6.4710,7.292,1.69500,0,7.194,6.805,9.146,4.429,1.677,3.762
2017-01-01 04:00:00.010,0,0,15.757,6.452,4.782,4.80,3.143,14.7610,10.040,7.030,...,5.4740,5.162,1.58500,0,6.533,5.952,9.319,4.192,1.614,3.993
2017-01-01 05:00:00.015,0,0,15.292,6.356,4.482,5.00,3.147,15.1560,10.147,6.043,...,5.5750,5.443,1.59200,0,6.816,5.790,8.417,2.948,2.208,3.183
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2017-12-31 20:00:43.770,0,0,17.613,6.635,9.339,7.92,9.038,12.4775,28.646,17.349,...,23.6007,3.980,4.59300,0,13.109,11.565,24.687,4.606,4.080,4.495
2017-12-31 21:00:43.775,0,0,17.181,6.455,5.926,7.68,10.707,11.8412,30.693,14.129,...,23.6007,2.319,4.63400,0,15.567,14.881,22.210,4.836,3.717,5.231
2017-12-31 22:00:43.780,0,0,16.932,6.196,5.675,7.40,10.144,12.3205,28.453,15.130,...,24.7513,3.468,3.58850,0,14.046,16.346,22.785,3.339,3.139,5.525


In [6]:
# Sum of every bus
sum_bus_feeders_P = np.sum(feeders_P, axis=0)

# Total sum of all buses
sum_feeders_P = np.sum(sum_bus_feeders_P)

# Average power consumption per household per year
avg_power_year = sum_feeders_P / 1120

# Average power consumption per household per hour
avg_power_hour = avg_power_year / 8760

print(f'Total Active Power in All Feeders: {sum_feeders_P:.2f} kW')
print(f'Average Active Power consumption for 1120 households (year) : {avg_power_year:.4f} kW')
print(f'Average Active Power consumption for 1120 households (hour): {avg_power_hour:.4f} kW')

Total Active Power in All Feeders: 13123541.50 kW
Average Active Power consumption for 1120 households (year) : 11717.4478 kW
Average Active Power consumption for 1120 households (hour): 1.3376 kW


In [7]:
# Increase the general load value
nonzero_buses_P = [bus for bus in feeders_P.columns if not feeders_P[bus].eq(0).all()]
feeders_P[nonzero_buses_P] = feeders_P[nonzero_buses_P] *1.0

nonzero_buses_Q = [bus for bus in feeders_Q.columns if not feeders_Q[bus].eq(0).all()]
feeders_Q[nonzero_buses_Q] = feeders_Q[nonzero_buses_Q] *1.0

print("All Load value has increased")

All Load value has increased


In [8]:
Loads_act = dss.utils.loads_to_dataframe()
Loads_act

,Name,Idx,Phases,Class,Model,NumCust,IsDelta,Rneut,Xneut,PF,...,kV,kW,kVABase,kvar,kWh,kWhDays,AllocationFactor,XfkVA,puSeriesRL,Sensor
load_1003,load_1003,1,3,1,1,1,False,-1.0,0.0,0.97,...,0.208,15.29,15.762887,3.832035,0.0,30.0,0.5,0.0,50.0,
load_1004,load_1004,2,3,1,1,1,False,-1.0,0.0,0.9,...,0.208,6.892,7.657778,3.337948,0.0,30.0,0.5,0.0,50.0,
load_1005,load_1005,3,3,1,1,1,False,-1.0,0.0,0.93,...,0.208,4.916,5.286022,1.942928,0.0,30.0,0.5,0.0,50.0,
load_1006,load_1006,4,1,1,1,1,True,-1.0,0.0,0.91,...,0.208,5.04,5.538462,2.296292,0.0,30.0,0.5,0.0,50.0,
load_1007,load_1007,5,1,1,1,1,True,-1.0,0.0,0.9,...,0.208,4.163,4.625556,2.016233,0.0,30.0,0.5,0.0,50.0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
load_3158,load_3158,190,2,1,1,1,False,-1.0,0.0,0.97,...,0.208,6.517,6.718557,1.633314,0.0,30.0,0.5,0.0,50.0,
load_3159,load_3159,191,2,1,1,1,False,-1.0,0.0,0.91,...,0.208,9.776,10.742857,4.454077,0.0,30.0,0.5,0.0,50.0,
load_3160,load_3160,192,2,1,1,1,False,-1.0,0.0,0.95,...,0.208,4.252,4.475789,1.397565,0.0,30.0,0.5,0.0,50.0,
load_3161,load_3161,193,2,1,1,1,False,-1.0,0.0,0.99,...,0.208,2.744,2.771717,0.390999,0.0,30.0,0.5,0.0,50.0,


In [9]:
# Extract the 2-phases load
#names_phases2 = Loads_act.loc[Loads_act['Phases'] == 2, 'Name']
# names_phases2 = Loads_act.loc[Loads_act['Phases'].isin([2, 3]), 'Name']
df_users = pd.read_excel(r'C:\Users\lfang7\Project\Distribution network\Add EV dataset\users number_1120.xlsx')
df_users['Load Name_lower'] = df_users['Load Name'].str.lower()
#filtered_df = df_users[df_users['Load Name_lower']]

In [10]:
df_users

,Load Name,Number of customers,EVs 0%,EVs 20%,EVs 40%,EVs 60%,EVs 80%,EVs 100%,Load Name_lower
0,Load_1003,17,0,6,8,13,10,17,load_1003
1,Load_1004,8,0,4,3,8,6,8,load_1004
2,Load_1005,8,0,3,6,2,8,8,load_1005
3,Load_1006,5,0,2,2,3,4,5,load_1006
4,Load_1007,3,0,1,2,3,3,3,load_1007
...,...,...,...,...,...,...,...,...,...
189,Load_3158,6,0,0,2,6,5,6,load_3158
190,Load_3159,8,0,1,2,5,2,8,load_3159
191,Load_3160,3,0,0,1,3,3,3,load_3160
192,Load_3161,2,0,0,2,2,2,2,load_3161


In [11]:
# Calculate the sum of each row
row_sums = feeders_P.sum(axis=1)

# Find the index of the row with the highest sum
max_row_index = row_sums.idxmax()

# Display the row with the highest sum
max_row_P = feeders_P.loc[max_row_index]

print(f"Row with the highest sum is at index: {max_row_index}")

Row with the highest sum is at index: 2017-07-12 13:00:23.095000


In [12]:
max_row_P_df = max_row_P.to_frame().T
max_row_P_df

,Bus 1001,Bus 1002,Bus 1003,Bus 1004,Bus 1005,Bus 1006,Bus 1007,Bus 1008,Bus 1009,Bus 1010,...,Bus 3153,Bus 3154,Bus 3155,Bus 3156,Bus 3157,Bus 3158,Bus 3159,Bus 3160,Bus 3161,Bus 3162
2017-07-12 13:00:23.095,0.0,0.0,66.155,29.835,47.733,19.36,3.915,10.4869,25.773,12.666,...,14.9431,8.13,9.514,0.0,17.722,15.606,23.042,4.883,7.097,8.494


In [13]:
# Display the row with the highest sum
max_row_Q = feeders_Q.loc[max_row_index]

# Convert the Series (max_row) to a DataFrame and transpose it
max_row_Q_df = max_row_Q.to_frame().T
max_row_Q_df

,Bus 1001,Bus 1002,Bus 1003,Bus 1004,Bus 1005,Bus 1006,Bus 1007,Bus 1008,Bus 1009,Bus 1010,...,Bus 3153,Bus 3154,Bus 3155,Bus 3156,Bus 3157,Bus 3158,Bus 3159,Bus 3160,Bus 3161,Bus 3162
2017-07-12 13:00:23.095,0.0,0.0,9.426577,8.701875,20.334173,2.758651,1.783727,2.628265,7.517125,4.163113,...,5.905891,3.937539,4.052947,0.0,5.168917,6.648128,10.498245,2.080149,1.011268,2.128797


In [14]:
charging_power = 10  # 5 10 15
ev_rate = [100]

base_P_series = max_row_P_df.iloc[0] 

result_indices = df_users['Load Name']
max_row_P_ev_df = pd.DataFrame(index=result_indices, columns=[f'Updated P {r}%' for r in ev_rate])
max_row_Q_ev_df = pd.DataFrame(index=result_indices, columns=[f'Updated Q {r}%' for r in ev_rate])

for adoption_rate in ev_rate:
    ev_col_name = f'EVs {adoption_rate}%'
    result_col_name = f'Updated P {adoption_rate}%'
    result_col_name_Q = f'Updated Q {adoption_rate}%'
    
    for i, row in df_users.iterrows():
        load_name = row['Load Name']  # e.g.: "Load_1003"
        bus_id = load_name.split('_')[1] # extract "1003"
        bus_col_name = f"Bus {bus_id}"   # construct "Bus 1003" to match the column names in max_row_P_df
        
        # Check whether this Bus exists in the base-load data
        if bus_col_name in base_P_series.index:
            # 1. Get the base load
            base_P = base_P_series[bus_col_name]
            
            # 2. Get the EV count under this penetration rate
            ev_count = row[ev_col_name]
            
            # 3. Calculate active power P after adding EV load
            additional_P = ev_count * charging_power
            updated_P = base_P + additional_P
            
            # 4. Calculate the corresponding reactive power Q
            # Generate a random power factor (0.9 ~ 0.95)
            pf = np.random.uniform(0.9, 0.95)
            
            if updated_P > 0:
                # Q = P * tan(arccos(pf))
                updated_Q = updated_P * np.tan(np.arccos(pf))
            else:
                updated_Q = 0
            
            # 5. Store in the result DataFrame
            max_row_P_ev_df.at[load_name, result_col_name] = updated_P
            max_row_Q_ev_df.at[load_name, result_col_name_Q] = updated_Q
            
        else:
            # If some points in df_users cannot be found in max_row_P (data mismatch case)
            # Fill with 0 or NaN; here 0 is used
            max_row_P_ev_df.at[load_name, result_col_name] = 0
            max_row_Q_ev_df.at[load_name, result_col_name_Q] = 0

# 5. Display the first few rows of the results for checking
print("Updated Active Power (P) with EVs:")
display(max_row_P_ev_df.head())

print("\nUpdated Reactive Power (Q) with EVs:")
display(max_row_Q_ev_df.head())

Updated Active Power (P) with EVs:


,Updated P 100%
Load Name,
Load_1003,236.155
Load_1004,109.835
Load_1005,127.733
Load_1006,69.36
Load_1007,33.915



Updated Reactive Power (Q) with EVs:


,Updated Q 100%
Load Name,
Load_1003,101.50569
Load_1004,37.052691
Load_1005,47.787542
Load_1006,27.427458
Load_1007,15.668629


In [15]:
# Assume charging_power, ev_rate, df_users, and baseMVA have already been defined above
charging_power = 10 
current_rate = 100  # corresponds to ev_rate=[100]

# --- Preparation: create a mapping dictionary from Bus ID to EV count to avoid repeated df_users queries in the loop ---
# The key is an integer Bus ID, e.g., 1003, and the value is the EV count
bus_ev_map = {}
for i, row in df_users.iterrows():
    load_name = row['Load Name']   # "Load_1003"
    try:
        bus_id = int(load_name.split('_')[1]) # extract 1003
        bus_ev_map[bus_id] = row[f'EVs {current_rate}%']
    except:
        continue

# --- Time slicing (unchanged) ---
peak_date_str = str(max_row_index.date())
# Note: this is converted to p.u.
target_P_df = feeders_P.loc[peak_date_str] / (baseMVA * 1000) 
target_Q_df = feeders_Q.loc[peak_date_str] / (baseMVA * 1000)

T = target_P_df.index

# --- Generate dictionaries and add EV load ---
P_load_dict = {}
Q_load_dict = {}

# Pre-generate random power factors to keep the EV power factor at the same node consistent within the same day; do not generate them inside the time loop
# Key is bus_id, value is tan(acos(pf))
bus_pf_tan_map = {} 
for bus_id in bus_ev_map:
    pf = np.random.uniform(0.9, 0.95)
    bus_pf_tan_map[bus_id] = np.tan(np.arccos(pf))

print("Start building the load dictionary and adding EV load...")

for time, row in target_P_df.iterrows():
    for col in target_P_df.columns:
        if "Bus" in str(col):
            # 1. Extract Bus ID
            try:
                bus = int(str(col).replace("Bus", "").strip())
            except ValueError:
                continue # Skip column names that cannot be parsed
            
            # 2. Get base load (p.u.)
            base_p_pu = row[col]
            base_q_pu = target_Q_df.at[time, col]
            
            # 3. Calculate EV load (p.u.)
            ev_p_pu = 0.0
            ev_q_pu = 0.0
            
            if bus in bus_ev_map:
                ev_count = bus_ev_map[bus]
                if ev_count > 0:
                    # Calculate active power (kW -> p.u.)
                    ev_p_kw = ev_count * charging_power
                    ev_p_pu = ev_p_kw / (baseMVA * 1000)
                    
                    # Calculate reactive power (kVar -> p.u.)
                    # Use the pre-generated power-factor relationship
                    ev_q_pu = ev_p_pu * bus_pf_tan_map[bus]

            # 4. Add and store in the dictionary
            P_load_dict[(bus, time)] = base_p_pu + ev_p_pu
            Q_load_dict[(bus, time)] = base_q_pu + ev_q_pu

print(f"Generated the load dictionary for {peak_date_str} and successfully added EV load (p.u.)")

Start building the load dictionary and adding EV load...
Generated the load dictionary for 2017-07-12 and successfully added EV load (p.u.)


In [16]:
P_load_dict

{(1001, Timestamp('2017-07-12 00:00:23.030000')): 0.0,
 (1002, Timestamp('2017-07-12 00:00:23.030000')): 0.0,
 (1003, Timestamp('2017-07-12 00:00:23.030000')): 0.0186025,
 (1004, Timestamp('2017-07-12 00:00:23.030000')): 0.0089899,
 (1005, Timestamp('2017-07-12 00:00:23.030000')): 0.0083039,
 (1006, Timestamp('2017-07-12 00:00:23.030000')): 0.005296,
 (1007, Timestamp('2017-07-12 00:00:23.030000')): 0.0034758000000000002,
 (1008, Timestamp('2017-07-12 00:00:23.030000')): 0.00806837,
 (1009, Timestamp('2017-07-12 00:00:23.030000')): 0.010298199999999999,
 (1010, Timestamp('2017-07-12 00:00:23.030000')): 0.0046295,
 (1011, Timestamp('2017-07-12 00:00:23.030000')): 0.0044829,
 (1012, Timestamp('2017-07-12 00:00:23.030000')): 0.004563,
 (1013, Timestamp('2017-07-12 00:00:23.030000')): 0.0021552,
 (1014, Timestamp('2017-07-12 00:00:23.030000')): 0.0011412,
 (1015, Timestamp('2017-07-12 00:00:23.030000')): 0.0012517000000000001,
 (1016, Timestamp('2017-07-12 00:00:23.030000')): 0.004563,
 (1

In [17]:
len(P_load_dict)

5736

In [18]:
voltage_df = pd.read_excel(r"C:\Users\lfang7\Project\Distribution network\Add EV code\bus1_voltage_mean.xlsx")
voltage_df["TimeStep"] = pd.to_datetime(voltage_df["TimeStep"])
voltage_df_filtered = voltage_df[voltage_df["TimeStep"].isin(T)].copy()

if len(voltage_df_filtered) == 0:
    raise ValueError(f"Cannot find the corresponding time points T in the voltage file. Please check whether the time formats in Excel and T are consistent.")
elif len(voltage_df_filtered) != len(T):
    print(f"Warning: the length of T ({len(T)}) and the length of the filtered voltage data ({len(voltage_df_filtered)}) do not fully match; duplicates or missing values may exist.")

bus1_voltage_mean = dict(zip(voltage_df_filtered["TimeStep"], voltage_df_filtered["Bus1 Voltage Mean (p.u.)"]))

print(f"Bus1 voltage dictionary has been generated with {len(bus1_voltage_mean)} time points.")
print(voltage_df_filtered["TimeStep"].head())

Bus1 voltage dictionary has been generated with 24 time points.
4607   2017-07-12 00:00:23.030
4608   2017-07-12 01:00:23.035
4609   2017-07-12 02:00:23.040
4610   2017-07-12 03:00:23.045
4611   2017-07-12 04:00:23.050
Name: TimeStep, dtype: datetime64[ns]


In [19]:
import pandas as pd

excel_path = r'C:\Users\lfang7\Project\Distribution network\Add EV dataset\price\houston_hourly_avg_price.xlsx'
df = pd.read_excel(excel_path)
df['Delivery Date'] = feeders_P.index

operation_price = dict(zip(df['Delivery Date'], df['Settlement Point Price']))
for t in list(operation_price)[:5]:
    print(f"{t}: {operation_price[t]}")


2017-01-01 01:00:00: 23.3575
2017-01-01 02:00:00: 21.465
2017-01-01 03:00:00.005000: 20.735
2017-01-01 04:00:00.010000: 20.27
2017-01-01 05:00:00.015000: 20.12


In [20]:
# without transformer impedance all 3-phase lines
gen = [(1, 0, 0, 300, -300, 1.02, 10, 1, 300, 0)]

branch = [
    (1, 1, 1001, 0.0001, 0.0001, 340),
    (2, 1, 2001, 0.0001, 0.0001, 340),
    (3, 1, 3001, 0.0001, 0.0001, 340),
    (4, 1001, 1002, 0.3369854539772726, 0.22045428125, 326),
    (5, 1002, 1003, 0.08049361363636363, 0.03148085227272727, 228),
    (6, 1002, 1004, 0.05377083333333333, 0.09379095416666668, 340),
    (7, 1004, 1005, 0.03320643939393939, 0.05792105946969697, 340),
    (8, 1005, 1006, 0.08841003787878787, 0.1542111456439394, 340),
    (9, 1006, 1007, 0.1685606060606061, 0.2940155303030303, 340),
    (10, 1006, 1008, 0.03826325757575758, 0.06674152537878789, 340),
    (11, 1008, 1009, 0.09119128787878786, 0.1590624018939394, 340),
    (12, 1009, 1010, 0.01424337121212121, 0.02484431231060606, 340),
    (13, 1009, 1011, 0.01298284090909091, 0.005077556818181819, 228),
    (14, 1011, 1012, 0.01298284090909091, 0.005077556818181819, 228),
    (15, 1012, 1013, 0.06621248863636364, 0.02589553977272728, 228),
    (16, 1013, 1014, 0.04500718181818181, 0.01760219696969697, 228),
    (17, 1014, 1015, 0.02726396590909091, 0.01066286931818182, 228),
    (18, 1013, 1016, 0.1012661590909091, 0.03960494318181819, 228),
    (19, 1016, 1017, 0.01536302840909091, 0.006008442234848485, 228),
    (20, 2001, 2002, 0.1576460431818182, 0.1031312916666667, 326),
    (21, 2002, 2003, 0.1079630681818182, 0.1883169471590909, 340),
    (22, 2003, 2004, 0.06725568181818181, 0.1173121965909091, 340),
    (23, 2004, 2005, 0.02671685606060606, 0.04660146155303031, 340),
    (24, 2005, 2006, 0.01222064393939394, 0.0213161259469697, 340),
    (25, 2006, 2007, 0.04439478882575758, 0.03341786079545454, 242),
    (26, 2007, 2008, 0.02704758522727273, 0.01057824337121212, 228),
    (27, 2007, 2009, 0.02917371837121212, 0.02196030852272727, 242),
    (28, 2006, 2010, 0.01432765151515152, 0.02499132007575758, 340),
    (29, 2010, 2011, 0.02317708333333333, 0.04042713541666667, 340),
    (30, 2011, 2012, 0.04239299242424242, 0.07394490587121211, 340),
    (31, 2012, 2013, 0.0001, 0.0001, 340),
    (32, 2013, 2014, 0.02258712121212121, 0.03939808106060606, 340),
    (33, 2014, 2015, 0.02596568181818182, 0.01015511363636364, 228),
    (34, 2014, 2016, 0.02166003787878788, 0.0377809956439394, 340),
    (35, 2016, 2017, 0.03765023863636363, 0.01472491477272727, 228),
    (36, 2017, 2018, 0.09672216477272727, 0.03782779829545455, 228),
    (37, 2013, 2019, 0.003792613636363636, 0.006615349431818181, 340),
    (38, 2019, 2020, 0.1389163977272727, 0.05432985795454547, 228),
    (39, 2019, 2021, 0.003624053030303031, 0.006321333901515153, 340),
    (40, 2021, 2022, 0.04435803977272727, 0.01734831912878788, 228),
    (41, 2022, 2023, 0.03245710227272727, 0.01269389204545454, 228),
    (42, 2023, 2024, 0.08828331818181819, 0.03452738636363636, 228),
    (43, 2024, 2025, 0.07162200568181817, 0.0280111884469697, 228),
    (44, 2021, 2026, 0.0001, 0.0001, 340),
    (45, 2026, 2027, 0.03109943181818182, 0.0542458653409091, 340),
    (46, 2027, 2028, 0.06166849431818182, 0.02411839488636364, 228),
    (47, 2028, 2029, 0.02834586931818182, 0.0110859990530303, 228),
    (48, 2029, 2030, 0.0357028125, 0.01396328125, 228),
    (49, 2030, 2031, 0.05301326704545455, 0.02073335700757576, 228),
    (50, 2027, 2032, 0.01255776515151515, 0.02190415700757576, 340),
    (51, 2032, 2033, 0.02435700757575757, 0.04248524412878787, 340),
    (52, 2033, 2034, 0.03352259564393939, 0.02523389488636363, 242),
    (53, 2033, 2035, 0.01348484848484848, 0.02352124242424242, 340),
    (54, 2035, 2036, 0.01348484848484848, 0.02352124242424242, 340),
    (55, 2036, 2037, 0.006742424242424242, 0.01176062121212121, 340),
    (56, 2037, 2038, 0.01618181818181818, 0.02822549090909091, 340),
    (57, 2038, 2039, 0.06742424242424241, 0.1176062121212121, 340),
    (58, 2039, 2040, 0.02604261363636364, 0.04542539943181818, 340),
    (59, 2040, 2041, 0.03227935606060606, 0.05630397405303032, 340),
    (60, 2039, 2042, 0.007573323863636364, 0.002961908143939395, 228),
    (61, 2042, 2043, 0.03418814772727272, 0.01337089962121212, 228),
    (62, 2043, 2044, 0.0740021931818182, 0.02894207386363636, 228),
    (63, 2044, 2045, 0.06989096022727273, 0.02733418087121212, 228),
    (64, 2045, 2046, 0.071405625, 0.0279265625, 228),
    (65, 2045, 2047, 0.04089594886363636, 0.01599430397727273, 228),
    (66, 2047, 2048, 0.05604259659090909, 0.02191812026515151, 228),
    (67, 2044, 2049, 0.02812948863636364, 0.01100137310606061, 228),
    (68, 2049, 2050, 0.02683120454545455, 0.01049361742424243, 228),
    (69, 2050, 2051, 0.03007691477272727, 0.01176300662878788, 228),
    (70, 2051, 2052, 0.02899501136363637, 0.01133987689393939, 228),
    (71, 2044, 2053, 0.1004006363636364, 0.0392664393939394, 228),
    (72, 2053, 2054, 0.03007691477272727, 0.01176300662878788, 228),
    (73, 2054, 2055, 0.03375538636363637, 0.01320164772727273, 228),
    (74, 2055, 2056, 0.04414165909090909, 0.01726369318181818, 228),
    (75, 2053, 2057, 0.1646656988636364, 0.0644003456439394, 228),
    (76, 2057, 2058, 0.0880669375, 0.03444276041666668, 228),
    (77, 2057, 2059, 0.1482207670454546, 0.05796877367424243, 228),
    (78, 2059, 2060, 0.06577972727272727, 0.02572628787878788, 228),
    (79, 3001, 3003, 0.2620240789772727, 0.17141490625, 326),
    (80, 3003, 3002, 0.01839235795454545, 0.007193205492424243, 228),
    (81, 3003, 3004, 0.01752683522727273, 0.006854701704545455, 228),
    (82, 3003, 3005, 0.2268149482954546, 0.1483812604166667, 326),
    (83, 3005, 3006, 0.02621117424242424, 0.0457194149621212, 340),
    (84, 3006, 3007, 0.03118371212121212, 0.05439287310606061, 340),
    (85, 3005, 3008, 0.01146212121212121, 0.01999305606060606, 340),
    (86, 3008, 3009, 0.02917371837121212, 0.02196030852272727, 242),
    (87, 3009, 3010, 0.0628775172348485, 0.04733060284090908, 242),
    (88, 3010, 3011, 0.04801885321969697, 0.03614584943181817, 242),
    (89, 3011, 3012, 0.05653540454545455, 0.04255662272727273, 242),
    (90, 3008, 3013, 0.03080454734848485, 0.0231879034090909, 242),
    (91, 3013, 3014, 0.05345494981060606, 0.04023783238636364, 242),
    (92, 3008, 3015, 0.02713825757575758, 0.04733650037878788, 340),
    (93, 3015, 3016, 0.03805267613636364, 0.02864388068181818, 242),
    (94, 3016, 3017, 0.04693163390151515, 0.03532745284090909, 242),
    (95, 3015, 3018, 0.03279778276515151, 0.02468829715909091, 242),
    (96, 3018, 3019, 0.05091810473484849, 0.03832824034090909, 242),
    (97, 3019, 3020, 0.0556293884469697, 0.04187462556818182, 242),
    (98, 3020, 3021, 0.05490457556818182, 0.04132902784090908, 242),
    (99, 3015, 3022, 0.02435700757575757, 0.04248524412878787, 340),
    (100, 3022, 3023, 0.03279778276515151, 0.02468829715909091, 242),
    (101, 3023, 3024, 0.05798503030303031, 0.04364781818181819, 242),
    (102, 3024, 3025, 0.05997826571969697, 0.04514821193181819, 242),
    (103, 3022, 3026, 0.02971732803030303, 0.02236950681818182, 242),
    (104, 3026, 3027, 0.06668278484848486, 0.05019499090909092, 242),
    (105, 3027, 3028, 0.0436699759469697, 0.03287226306818182, 242),
    (106, 3028, 3029, 0.05744142064393939, 0.04323861988636363, 242),
    (107, 3022, 3030, 0.02562121212121212, 0.0446903606060606, 340),
    (108, 3030, 3035, 0.008680871212121213, 0.01514179981060606, 340),
    (109, 3035, 3036, 0.02283996212121212, 0.0398391043560606, 340),
    (110, 3036, 3037, 0.01146212121212121, 0.01999305606060606, 340),
    (111, 3037, 3038, 0.02123863636363637, 0.03704595681818182, 340),
    (112, 3038, 3039, 0.01980587121212122, 0.03454682481060606, 340),
    (113, 3039, 3053, 0.0221657196969697, 0.03866304223484849, 340),
    (114, 3053, 3054, 0.06058659090909091, 0.02369526515151516, 228),
    (115, 3053, 3055, 0.2888682102272727, 0.1129756392045455, 228),
    (116, 3055, 3056, 0.02769672727272728, 0.01083212121212121, 228),
    (117, 3056, 3057, 0.03851576136363637, 0.01506341856060606, 228),
    (118, 3057, 3058, 0.03851576136363637, 0.01506341856060606, 228),
    (119, 3058, 3059, 0.05928830681818181, 0.02318750946969697, 228),
    (120, 3059, 3060, 0.0166613125, 0.006516197916666668, 228),
    (121, 3060, 3061, 0.03981404545454546, 0.01557117424242424, 228),
    (122, 3055, 3062, 0.09585664204545455, 0.03748929450757577, 228),
    (123, 3062, 3063, 0.042843375, 0.0167559375, 228),
    (124, 3063, 3064, 0.03397176704545454, 0.01328627367424243, 228),
    (125, 3064, 3065, 0.04435803977272727, 0.01734831912878788, 228),
    (126, 3065, 3066, 0.02661482386363637, 0.01040899147727273, 228),
    (127, 3066, 3067, 0.05928830681818181, 0.02318750946969697, 228),
    (128, 3030, 3040, 0.02798106060606061, 0.04880657803030303, 340),
    (129, 3040, 3044, 0.02609326363636364, 0.01964151818181818, 242),
    (130, 3044, 3045, 0.07121286534090909, 0.05360497670454545, 242),
    (131, 3040, 3041, 0.02192558958333333, 0.01650433125, 242),
    (132, 3041, 3042, 0.05943465606060606, 0.04473901363636364, 242),
    (133, 3042, 3043, 0.07574294583333334, 0.05701496249999999, 242),
    (134, 3040, 3046, 0.0277282196969697, 0.04836555473484849, 340),
    (135, 3046, 3047, 0.01845738636363637, 0.03219470056818182, 340),
    (136, 3046, 3048, 0.01095643939393939, 0.01911100946969697, 340),
    (137, 3048, 3049, 0.01904734848484848, 0.03322375492424243, 340),
    (138, 3049, 3050, 0.01407481060606061, 0.02455029678030303, 340),
    (139, 3050, 3051, 0.03034090909090909, 0.05292279545454545, 340),
    (140, 3051, 3052, 0.02460984848484848, 0.04292626742424244, 340),
    (141, 3030, 3031, 0.02048011363636364, 0.03572288693181819, 340),
    (142, 3031, 3032, 0.02115435606060606, 0.0368989490530303, 340),
    (143, 3032, 3033, 0.03270075757575758, 0.05703901287878788, 340),
    (144, 3033, 3034, 0.02166003787878788, 0.0377809956439394, 340),
    (145, 3034, 3068, 0.0138219696969697, 0.02410927348484848, 340),
    (146, 3068, 3069, 0.0277282196969697, 0.04836555473484849, 340),
    (147, 3069, 3070, 0.02315273295454545, 0.009054976325757575, 228),
    (148, 3070, 3071, 0.03050967613636364, 0.01193225852272727, 228),
    (149, 3071, 3072, 0.01947426136363636, 0.007616335227272728, 228),
    (150, 3069, 3073, 0.001685606060606061, 0.002940155303030303, 340),
    (151, 3073, 3074, 0.02865530303030303, 0.04998264015151516, 340),
    (152, 3068, 3075, 0.01584469696969697, 0.02763745984848485, 340),
    (153, 3075, 3076, 0.0001, 0.0001, 340),
    (154, 3076, 3077, 0.07391382575757577, 0.1289258100378788, 340),
    (155, 3077, 3078, 0.006742424242424242, 0.01176062121212121, 340),
    (156, 3076, 3079, 0.01609753787878788, 0.02807848314393939, 340),
    (157, 3079, 3080, 0.05090530303030303, 0.08879269015151514, 340),
    (158, 3080, 3081, 0.01643465909090909, 0.02866651420454545, 340),
    (159, 3080, 3082, 0.04551136363636363, 0.07938419318181819, 340),
    (160, 3082, 3083, 0.02704758522727273, 0.01057824337121212, 228),
    (161, 3083, 3084, 0.04241061363636364, 0.01658668560606061, 228),
    (162, 3084, 3085, 0.1114360511363636, 0.04358236268939394, 228),
    (163, 3085, 3086, 0.07313667045454544, 0.02860357007575758, 228),
    (164, 3086, 3087, 0.05171498295454545, 0.02022560132575758, 228),
    (165, 3087, 3088, 0.09087988636363636, 0.03554289772727272, 228),
    (166, 3088, 3089, 0.02423463636363636, 0.009478106060606062, 228),
    (167, 3089, 3090, 0.1222550852272727, 0.04781366003787879, 228),
    (168, 3090, 3091, 0.04803651136363636, 0.01878696022727273, 228),
    (169, 3082, 3092, 0.04003042613636364, 0.01565580018939394, 228),
    (170, 3092, 3093, 0.04976755681818182, 0.0194639678030303, 228),
    (171, 3093, 3094, 0.02704758522727273, 0.01057824337121212, 228),
    (172, 3094, 3095, 0.01406474431818182, 0.005500686553030303, 228),
    (173, 3095, 3096, 0.06404868181818181, 0.02504928030303031, 228),
    (174, 3096, 3097, 0.02531653977272727, 0.009901235795454547, 228),
    (175, 3092, 3098, 0.06145211363636364, 0.02403376893939394, 228),
    (176, 3098, 3099, 0.07962809090909091, 0.03114234848484849, 228),
    (177, 3092, 3100, 0.01969064204545454, 0.007700961174242424, 228),
    (178, 3100, 3101, 0.03397176704545454, 0.01328627367424243, 228),
    (179, 3101, 3102, 0.0486856534090909, 0.01904083806818182, 228),
    (180, 3102, 3103, 0.02899501136363637, 0.01133987689393939, 228),
    (181, 3100, 3104, 0.04825289204545455, 0.01887158617424242, 228),
    (182, 3104, 3105, 0.02596568181818182, 0.01015511363636364, 228),
    (183, 3105, 3106, 0.0391649034090909, 0.01531729640151515, 228),
    (184, 3082, 3107, 0.1659372255681818, 0.10855534375, 326),
    (185, 3107, 3108, 0.1008333977272727, 0.03943569128787879, 228),
    (186, 3108, 3109, 0.02942777272727273, 0.01150912878787879, 228),
    (187, 3109, 3110, 0.03288986363636363, 0.01286314393939394, 228),
    (188, 3110, 3111, 0.05972106818181818, 0.02335676136363636, 228),
    (189, 3111, 3112, 0.07097286363636364, 0.02775731060606061, 228),
    (190, 3107, 3113, 0.05301326704545455, 0.02073335700757576, 228),
    (191, 3113, 3114, 0.0322407215909091, 0.01260926609848485, 228),
    (192, 3114, 3115, 0.07162200568181817, 0.0280111884469697, 228),
    (193, 3113, 3116, 0.09866959090909092, 0.03858943181818182, 228),
    (194, 3116, 3117, 0.02423463636363636, 0.009478106060606062, 228),
    (195, 3107, 3118, 0.2574809653409091, 0.1684428229166667, 326),
    (196, 3118, 3119, 0.1004006363636364, 0.0392664393939394, 228),
    (197, 3119, 3120, 0.07573323863636364, 0.02961908143939394, 228),
    (198, 3120, 3121, 0.04457442045454546, 0.01743294507575758, 228),
    (199, 3121, 3122, 0.03829938068181818, 0.01497879261363636, 228),
    (200, 3119, 3123, 0.01969064204545454, 0.007700961174242424, 228),
    (201, 3123, 3124, 0.01449750568181818, 0.005669938446969697, 228),
    (202, 3124, 3125, 0.03267348295454545, 0.01277851799242424, 228),
    (203, 3125, 3126, 0.07421857386363635, 0.02902669981060606, 228),
    (204, 3126, 3127, 0.05063307954545455, 0.01980247159090909, 228),
    (205, 3127, 3128, 0.05149860227272726, 0.02014097537878788, 228),
    (206, 3128, 3129, 0.05258050568181818, 0.02056410511363636, 228),
    (207, 3129, 3130, 0.04976755681818182, 0.0194639678030303, 228),
    (208, 3130, 3131, 0.04197785227272727, 0.01641743371212121, 228),
    (209, 3118, 3132, 0.1084067215909091, 0.04239759943181818, 228),
    (210, 3132, 3133, 0.03137519886363636, 0.01227076231060606, 228),
    (211, 3133, 3134, 0.0608029715909091, 0.02377989109848485, 228),
    (212, 3134, 3135, 0.05193136363636364, 0.02031022727272727, 228),
    (213, 3135, 3136, 0.07941171022727272, 0.03105772253787879, 228),
    (214, 3133, 3137, 0.05452793181818182, 0.02132573863636364, 228),
    (215, 3137, 3138, 0.07594961931818181, 0.02970370738636364, 228),
    (216, 3138, 3139, 0.06253401704545454, 0.02445689867424243, 228),
    (217, 3107, 3140, 0.05517707386363636, 0.02157961647727272, 228),
    (218, 3140, 3141, 0.02899501136363637, 0.01133987689393939, 228),
    (219, 3141, 3142, 0.04176147159090909, 0.01633280776515152, 228),
    (220, 3142, 3143, 0.04370889772727272, 0.01709444128787879, 228),
    (221, 3140, 3148, 0.03180796022727272, 0.01244001420454545, 228),
    (222, 3148, 3149, 0.0547443125, 0.02141036458333334, 228),
    (223, 3149, 3150, 0.03007691477272727, 0.01176300662878788, 228),
    (224, 3150, 3151, 0.03375538636363637, 0.01320164772727273, 228),
    (225, 3151, 3152, 0.05625897727272727, 0.02200274621212121, 228),
    (226, 3152, 3153, 0.03007691477272727, 0.01176300662878788, 228),
    (227, 3153, 3154, 0.02574930113636363, 0.01007048768939394, 228),
    (228, 3154, 3155, 0.1092722443181818, 0.04273610321969697, 228),
    (229, 3140, 3156, 0.06902543750000001, 0.02699567708333333, 228),
    (230, 3156, 3144, 0.02639844318181818, 0.01032436553030303, 228),
    (231, 3144, 3145, 0.03851576136363637, 0.01506341856060606, 228),
    (232, 3145, 3146, 0.03851576136363637, 0.01506341856060606, 228),
    (233, 3146, 3147, 0.03851576136363637, 0.01506341856060606, 228),
    (234, 3156, 3157, 0.03310624431818182, 0.01294776988636364, 228),
    (235, 3157, 3158, 0.06383230113636364, 0.02496465435606061, 228),
    (236, 3158, 3159, 0.05560983522727274, 0.02174886837121212, 228),
    (237, 3159, 3160, 0.05258050568181818, 0.02056410511363636, 228),
    (238, 3160, 3161, 0.05128222159090909, 0.02005634943181818, 228),
    (239, 3161, 3162, 0.038083, 0.01489416666666667, 228)
]

In [21]:
gen = pd.DataFrame(gen, columns=['bus', 'Pg', 'Qg', 'Qmax', 'Qmin', 'Vg', 'mBase', 'status', 'Pmax', 'Pmin'])
branch = pd.DataFrame(branch, columns=['branch_id', 'fbus', 'tbus', 'r', 'x', 'rateA'])

bus_from_load = {bus for (bus, _) in P_load_dict.keys()}
bus_from_branch = set(branch['fbus'].tolist() + branch['tbus'].tolist())
bus_list = sorted(bus_from_load.union(bus_from_branch))

gen['Pg'] = gen['Pg'] / baseMVA
gen['Qg'] = gen['Qg'] / baseMVA
gen['Pmax'] = gen['Pmax'] / baseMVA
gen['Pmin'] = gen['Pmin'] / baseMVA
gen['Qmax'] = gen['Qmax'] / baseMVA
gen['Qmin'] = gen['Qmin'] / baseMVA

ij = list(zip(branch['fbus'], branch['tbus']))
r_ij = dict(zip(zip(branch['fbus'], branch['tbus']), branch['r'] / Zb))
x_ij = dict(zip(zip(branch['fbus'], branch['tbus']), branch['x'] / Zb))

upStream = {Node: branch[branch['tbus'] == Node]['fbus'].astype(int).tolist() for Node in bus_list}
downStream = {Node: branch[branch['fbus'] == Node]['tbus'].astype(int).tolist() for Node in bus_list}

In [22]:
#commen_num = list(set(bus['bus'].tolist()) - set(gen['bus'].tolist()) - set([14]))
commen_num = list(set(bus_list) - set(gen['bus'].tolist()))
gen_indexed = gen.set_index('bus')
Pmax_dict = {(i, t): gen_indexed.loc[i, 'Pmax'] for i in gen['bus'] for t in T}
Qmax_dict = {(i, t): gen_indexed.loc[i, 'Qmax'] for i in gen['bus'] for t in T}
Qmin_dict = {(i, t): gen_indexed.loc[i, 'Qmin'] for i in gen['bus'] for t in T}

model = Model('DistFlow')

GP_i = model.addVars(gen['bus'], T, lb=0, ub=Pmax_dict, name='GP_i') 
GQ_i = model.addVars(gen['bus'], T, lb=Qmin_dict, ub=Qmax_dict, name='GQ_i')  
P_ij = model.addVars(ij, T, lb=-GRB.INFINITY, name='P_ij')  
Q_ij = model.addVars(ij, T, lb=-GRB.INFINITY, name='Q_ij')  
#l_ij = model.addVars(ij, T, lb=0, ub=I_max, name='l_ij') 
l_ij = model.addVars(ij, T, lb=0, name='l_ij') 
v_i = model.addVars(bus_list, T, lb=V_min, ub=V_max, name='v_i')  

line_limit_dict = dict(zip(zip(branch['fbus'], branch['tbus']), branch['rateA']))

Set parameter Username
Academic license - for non-commercial use only - expires 2026-10-03


In [23]:
def load_line_lengths(file_path='length.xlsx'):
    print(f"\n>>> [Data Load] reading from {file_path} line length data...")
    length_dict = {}
    
    try:
        df = pd.read_excel(file_path, engine='openpyxl')
        
        # Iterate through each row
        for i in range(len(df)):
            # Assume the Excel structure: second column = From, third column = To, last column = Length_ft
            u = int(df.iloc[i, 1])      # second column (From)
            v = int(df.iloc[i, 2])      # third column (To)
            l_ft = float(df.iloc[i, -1]) # last column (Length ft)
            
            # Store in the dictionary with bidirectional indexing
            length_dict[(u, v)] = l_ft
            length_dict[(v, u)] = l_ft 
            
        print(f"Successfully loaded {len(length_dict)//2} line-length records.")
        return length_dict

    except Exception as e:
        print(f"[Error] Failed to read the length file: {e}")
        print("Default values will be used or length calculation will be skipped.")
        return {}

In [24]:
import pandas as pd
import ast
import math

def update_top_k_bottlenecks_topology(k, line_limit_dict, r_ij, x_ij, length_dict, csv_path='Result_Line_IAS_6.9_DDCP.csv'):
    """
    Perform line capacity upgrades based on topological order.
    Correction: use ohms (Ω) consistently for ratio calculation and dictionary updates to ensure unit consistency.
    """
    print(f"\n>>> [Strategy] Execute Top-{k} capacity upgrades based on topological order (Source->Load)...")
    
    # 0. Create copies to preserve the original data
    r_ij_new = r_ij.copy()
    x_ij_new = x_ij.copy()
    line_limit_new = line_limit_dict.copy()
    
    # Unit conversion factor (feet -> kilometers)
    FT_TO_KM = 0.0003048
    total_upgrade_cost = 0.0
    
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print("Error: diagnostic file not found.")
        return

    # ==============================================================================
    # 1. Confirm database (units: Ω/km, $/km)
    # ==============================================================================
    CABLE_DB = [
        {'Size': '120mm2',   'Limit': 330,  'R_unit': 0.196, 'X_unit': 0.113, 'Cost': 98000},
        {'Size': '150mm2',   'Limit': 370,  'R_unit': 0.159, 'X_unit': 0.109, 'Cost': 115000},
        {'Size': '185mm2',   'Limit': 420,  'R_unit': 0.127, 'X_unit': 0.106, 'Cost': 130000},
        {'Size': '240mm2',   'Limit': 485,  'R_unit': 0.098, 'X_unit': 0.101, 'Cost': 155000},
        {'Size': '300mm2',   'Limit': 540,  'R_unit': 0.079, 'X_unit': 0.098, 'Cost': 180000},
        {'Size': '400mm2',   'Limit': 610,  'R_unit': 0.063, 'X_unit': 0.094, 'Cost': 220000},
        {'Size': '500mm2',   'Limit': 690,  'R_unit': 0.051, 'X_unit': 0.091, 'Cost': 260000},
        {'Size': '630mm2',   'Limit': 780,  'R_unit': 0.040, 'X_unit': 0.089, 'Cost': 310000},
        {'Size': '800mm2',   'Limit': 860,  'R_unit': 0.031, 'X_unit': 0.087, 'Cost': 380000},
        {'Size': '2x500mm2', 'Limit': 1200, 'R_unit': 0.026, 'X_unit': 0.046, 'Cost': 460000}
    ]

    BREAKER_DB = [
        {'Size': 'CB_630A',  'Limit': 630,  'Cost': 18000},  
        {'Size': 'CB_1250A', 'Limit': 1250, 'Cost': 24000}
    ]

    # Helper functions
    def select_new_cable(req_amps):
        target = req_amps * 1.05 
        for c in CABLE_DB:
            if c['Limit'] >= target: return c
        return CABLE_DB[-1]

    def select_new_breaker(req_amps):
        target = req_amps * 1.05 
        for b in BREAKER_DB:
            if b['Limit'] >= target: return b
        return BREAKER_DB[-1]

    # ==========================================
    # 2. Topological screening
    # ==========================================
    grouped = df.groupby('Branch').agg({
        'Actual (A)': 'max',
        'Limit (A)': 'first',
        'Overload (%)': 'max'
    }).reset_index()

    overloaded_lines = grouped[grouped['Overload (%)'] > 0].copy()

    if overloaded_lines.empty:
        print("No overloaded lines found in the current case.")
        return line_limit_new, r_ij_new, x_ij_new

    overloaded_lines['From_Node'] = overloaded_lines['Branch'].apply(lambda x: ast.literal_eval(x)[0])
    target_lines = overloaded_lines.sort_values(by=['From_Node', 'Actual (A)'], ascending=[True, False]).head(k)

    print(f"Selected the top {len(target_lines)} topological bottlenecks:")
    print(f"{'Branch':<15} | {'Load(A)':<8} | {'Type':<8} | {'Upgrade To':<12} | {'Old Z(Ω)':<10} | {'New Z(Ω)':<10} | {'Ratio':<8} | {'Cost($)':<10}")
    print("-" * 100)

    updated_set = set()

    for _, row in target_lines.iterrows():
        try:
            u, v = ast.literal_eval(row['Branch'])
            req_amps = row['Actual (A)']
            
            # Determine equipment type
            current_r = r_ij.get((u, v), r_ij.get((v, u), 1.0))
            is_switch = (u == 1) or (current_r < 1e-4)

            # ---------------------------------------------------------
            # 3.1 Get old impedance (Z_old_ohm)
            # Assume the input r_ij is in ohms
            # ---------------------------------------------------------
            old_r_pu = r_ij.get((u, v), r_ij.get((v, u), 0))
            old_r = old_r_pu *Zb
            old_x_pu = x_ij.get((u, v), x_ij.get((v, u), 0))
            old_x=  old_x_pu *Zb
            z_old_ohm = math.sqrt(old_r**2 + old_x**2)

            if is_switch:
                # --- Breaker upgrade ---
                new_breaker = select_new_breaker(req_amps)
                
                # Update Limit
                line_limit_new[(u, v)] = new_breaker['Limit']
                if (v, u) in line_limit_new: line_limit_new[(v, u)] = new_breaker['Limit']
                
                cost = new_breaker['Cost']
                total_upgrade_cost += cost

                print(f"{str((u,v)):<15} | {req_amps:<8.1f} | {'CB':<8} | {new_breaker['Size']:<12} | {z_old_ohm:<10.5f} | {z_old_ohm:<10.5f} | {'100%':<8} | {cost:<10.0f}")

            else:
                # --- Cable upgrade ---
                new_cable = select_new_cable(req_amps)
                
                # 1. Update Limit
                line_limit_new[(u, v)] = new_cable['Limit']
                if (v, u) in line_limit_new: line_limit_new[(v, u)] = new_cable['Limit']

                # 2. Calculate new impedance (Z_new_ohm)
                length_ft = length_dict.get((u, v)) or length_dict.get((v, u))
                
                if length_ft:
                    length_km = length_ft * FT_TO_KM
                    
                    # Key modification: directly calculate the ohm value without dividing by Z_base
                    # Z_ohm = Z_unit(Ω/km) * Length(km)
                    new_r_ohm = new_cable['R_unit'] * length_km
                    new_x_ohm = new_cable['X_unit'] * length_km
                    
                    z_new_ohm = math.sqrt(new_r_ohm**2 + new_x_ohm**2)
                    
                    # Update the impedance dictionary while keeping ohm units
                    if (u, v) in r_ij_new:
                        r_ij_new[(u, v)] = new_r_ohm
                        x_ij_new[(u, v)] = new_x_ohm
                    if (v, u) in r_ij_new:
                        r_ij_new[(v, u)] = new_r_ohm
                        x_ij_new[(v, u)] = new_x_ohm
                        
                    # 3. Calculate the ratio (Z_new_ohm / Z_old_ohm)
                    # The numerator and denominator now use consistent units, so the ratio is correct
                    if z_old_ohm > 1e-9:
                        ratio = z_new_ohm / z_old_ohm
                    else:
                        ratio = 0.0 

                    if (u, v) in r_ij_new:
                        r_ij_new[(u, v)] = new_r_ohm/Zb
                        x_ij_new[(u, v)] = new_x_ohm/Zb
                    if (v, u) in r_ij_new:
                        r_ij_new[(v, u)] = new_r_ohm/Zb
                        x_ij_new[(v, u)] = new_x_ohm/Zb
                        
                    # 4. Calculate cost
                    cost = new_cable['Cost'] * length_km
                    total_upgrade_cost += cost
                    
                    print(f"{str((u,v)):<15} | {req_amps:<8.1f} | {'Cable':<8} | {new_cable['Size']:<12} | {z_old_ohm:<10.5f} | {z_new_ohm:<10.5f} | {ratio:<8.2%} | {cost:<10.0f}")
                
                else:
                    print(f"{str((u,v)):<15} | {req_amps:<8.1f} | {'WARN':<8} | {'Missing Len':<12} | {'-':<10} | {'-':<10} | {'-':<8} | {'0':<10}")

            updated_set.add((u, v))
            updated_set.add((v, u))
            
        except Exception as e:
            print(f"Error on {row['Branch']}: {e}")

    print("-" * 100)
    print(f">>> Topological capacity upgrade completed. Estimated total investment: ${total_upgrade_cost:,.2f}")
    
    return line_limit_new, r_ij_new, x_ij_new
length_dict = load_line_lengths()
update_top_k_bottlenecks_topology(20, line_limit_dict, r_ij, x_ij, length_dict, csv_path='Result_Line_IAS_6.9_DDCP.csv')


>>> [Data Load] reading from length.xlsx line length data...
Successfully loaded 233 line-length records.

>>> [Strategy] Execute Top-20 capacity upgrades based on topological order (Source->Load)...
Selected the top 20 topological bottlenecks:
Branch          | Load(A)  | Type     | Upgrade To   | Old Z(Ω)   | New Z(Ω)   | Ratio    | Cost($)   
----------------------------------------------------------------------------------------------------
(1, 3001)       | 933.1    | CB       | CB_1250A     | 0.00014    | 0.00014    | 100%     | 24000     
(1, 2001)       | 393.4    | CB       | CB_630A      | 0.00014    | 0.00014    | 100%     | 18000     
(2001, 2002)    | 393.4    | Cable    | 185mm2       | 0.18838    | 0.06998    | 37.15%   | 54998     
(3001, 3003)    | 979.8    | Cable    | 2x500mm2     | 0.31311    | 0.03716    | 11.87%   | 323460    
(3003, 3005)    | 950.2    | Cable    | 2x500mm2     | 0.27104    | 0.03216    | 11.87%   | 279995    
(3005, 3008)    | 801.7    | Cable 

({(1, 1001): 340,
  (1, 2001): 630,
  (1, 3001): 1250,
  (1001, 1002): 326,
  (1002, 1003): 228,
  (1002, 1004): 340,
  (1004, 1005): 340,
  (1005, 1006): 340,
  (1006, 1007): 340,
  (1006, 1008): 340,
  (1008, 1009): 340,
  (1009, 1010): 340,
  (1009, 1011): 228,
  (1011, 1012): 228,
  (1012, 1013): 228,
  (1013, 1014): 228,
  (1014, 1015): 228,
  (1013, 1016): 228,
  (1016, 1017): 228,
  (2001, 2002): 420,
  (2002, 2003): 340,
  (2003, 2004): 340,
  (2004, 2005): 340,
  (2005, 2006): 340,
  (2006, 2007): 242,
  (2007, 2008): 228,
  (2007, 2009): 242,
  (2006, 2010): 340,
  (2010, 2011): 340,
  (2011, 2012): 340,
  (2012, 2013): 340,
  (2013, 2014): 340,
  (2014, 2015): 228,
  (2014, 2016): 340,
  (2016, 2017): 228,
  (2017, 2018): 228,
  (2013, 2019): 340,
  (2019, 2020): 228,
  (2019, 2021): 340,
  (2021, 2022): 228,
  (2022, 2023): 228,
  (2023, 2024): 228,
  (2024, 2025): 228,
  (2021, 2026): 340,
  (2026, 2027): 340,
  (2027, 2028): 228,
  (2028, 2029): 228,
  (2029, 2030): 228,


In [25]:
line_limit_new, r_ij_new, x_ij_new = update_top_k_bottlenecks_topology(20, line_limit_dict, r_ij, x_ij, length_dict, csv_path='Result_Line_IAS_6.9_DDCP.csv')


>>> [Strategy] Execute Top-20 capacity upgrades based on topological order (Source->Load)...
Selected the top 20 topological bottlenecks:
Branch          | Load(A)  | Type     | Upgrade To   | Old Z(Ω)   | New Z(Ω)   | Ratio    | Cost($)   
----------------------------------------------------------------------------------------------------
(1, 3001)       | 933.1    | CB       | CB_1250A     | 0.00014    | 0.00014    | 100%     | 24000     
(1, 2001)       | 393.4    | CB       | CB_630A      | 0.00014    | 0.00014    | 100%     | 18000     
(2001, 2002)    | 393.4    | Cable    | 185mm2       | 0.18838    | 0.06998    | 37.15%   | 54998     
(3001, 3003)    | 979.8    | Cable    | 2x500mm2     | 0.31311    | 0.03716    | 11.87%   | 323460    
(3003, 3005)    | 950.2    | Cable    | 2x500mm2     | 0.27104    | 0.03216    | 11.87%   | 279995    
(3005, 3008)    | 801.7    | Cable    | 800mm2       | 0.02305    | 0.00383    | 16.61%   | 15752     
(3008, 3015)    | 767.6    | Cable    |

In [26]:
# This step must be included in the model code for Limit to take effect
print("Adding line capacity constraints...")

# Calculate base current for converting amperes to p.u.
I_base = (baseMVA * 1000) / (math.sqrt(3) * 6.9) # assume 6.9 kV
r_ij = r_ij_new
x_ij = x_ij_new

for (i, j) in ij:
    # Read from the dictionary; if the dictionary was modified by the update function, the new value will be read here
    current_limit_amps = line_limit_new.get((i, j)) 
    print(current_limit_amps)
    # Convert to squared p.u.
    limit_sq_pu = (current_limit_amps / I_base) ** 2

    
    for t in T:
        model.addConstr(l_ij[i, j, t] <= limit_sq_pu)

Adding line capacity constraints...
340
630
1250
326
228
340
340
340
340
340
340
340
228
228
228
228
228
228
228
420
340
340
340
340
242
228
242
340
340
340
340
340
228
340
228
228
340
228
340
228
228
228
228
340
340
228
228
228
228
340
340
242
340
340
340
340
340
340
340
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
1200
228
228
1200
340
340
860
242
242
242
242
242
242
860
242
242
242
242
242
242
780
242
242
242
242
242
242
242
780
340
340
340
340
340
340
228
228
228
228
228
228
228
228
228
228
228
228
228
228
340
242
242
242
242
242
340
340
340
340
340
340
340
610
610
610
540
540
340
228
228
228
340
340
540
630
340
340
485
485
340
485
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
370
228
228
228
228
228
228
228
228
228
228
326
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228
228


In [27]:
len(commen_num)

239

In [28]:
import pandas as pd
from gurobipy import GRB

# ==========================================
# 1. Extract the optimal BESS results and convert them into constant dictionaries
# ==========================================
# Assume baseMVA = 10, then base_kW = 10000
base_kW = 10000.0  # Make sure this value is consistent with the previous system base value

# Original optimal capacity results (unit: kW or kWh)
# optimal_bess_kw = {
#     2052: 400.848732,
#     2058: 1846.960601,
#     3118: 2000.000000,
#     3132: 1996.998048,
#     3158: 1937.499945
# }

optimal_bess_kw = {
    2058: 995.930619
}

# Convert to per-unit values (p.u.) for model use
optimal_bess_pu = {bus: cap / base_kW for bus, cap in optimal_bess_kw.items()}

# [Optimization] Now variables and constraints are generated only for buses with installed BESS
installed_buses = list(optimal_bess_pu.keys())
print(f"The current operation model will dispatch BESS only at the following buses: {installed_buses}")

# ==========================================
# 2. Operation parameter definitions
# ==========================================
SOC_lower_percent = 0.1
SOC_upper_percent = 0.9
eta_ch = 0.95  
eta_dis = 0.95 
Delta_t = 1     
C_rate = 0.5
INVERTER_OVERSIZE = 1.2

# Prepare time data
T_dt = list(pd.to_datetime(T))
num_time_steps = len(T_dt)

# ==========================================
# 3. Define operation variables only for installed_buses
# ==========================================
# Note: E_cap and x_build variables are removed because they are now constants

# Charging and discharging power variables
P_charge = model.addVars(T_dt, installed_buses, lb=0, name='P_charge')
P_discharge = model.addVars(T_dt, installed_buses, lb=0, name='P_discharge')
Q_inj = model.addVars(T_dt, installed_buses, lb=0, name='Q_inj')
Q_abs = model.addVars(T_dt, installed_buses, lb=0, name='Q_abs')

# Real-time energy variables; upper bounds are set by constants in the specific constraints instead of here
E = model.addVars(T_dt, installed_buses, lb=0, name='E_stored')

# Mutually exclusive binary state variables
y_charge = model.addVars(T_dt, installed_buses, vtype=GRB.BINARY, name='y_charge')
y_discharge = model.addVars(T_dt, installed_buses, vtype=GRB.BINARY, name='y_discharge')
y_inj = model.addVars(T_dt, installed_buses, vtype=GRB.BINARY, name='y_inj')
y_abs = model.addVars(T_dt, installed_buses, vtype=GRB.BINARY, name='y_abs')

# ==========================================
# 4. Add BESS operation constraints
# ==========================================
for b in installed_buses:
    # Get the fixed capacity constant for this bus
    cap_pu = optimal_bess_pu[b]
    
    # Dynamic energy balance constraint across time steps
    for idx in range(1, num_time_steps):
        t_prev = T_dt[idx - 1]
        t_curr = T_dt[idx]
        model.addConstr(
            E[t_curr, b] == E[t_prev, b] + eta_ch * P_charge[t_curr, b] - (1/eta_dis * P_discharge[t_curr, b]),
            name=f'E_dynamic_{b}_{t_curr.strftime("%Y%m%d_%H%M")}'
        )

    for t in T_dt:
        # SOC upper and lower bound constraints using fixed cap_pu
        model.addConstr(E[t, b] >= SOC_lower_percent * cap_pu, name=f'E_min_{b}_{t}')
        model.addConstr(E[t, b] <= SOC_upper_percent * cap_pu, name=f'E_max_{b}_{t}')
        
        # Charge/discharge mutual-exclusion constraint
        model.addConstr(y_charge[t, b] + y_discharge[t, b] <= 1, name=f'charge_exclusive_{b}_{t}')
        model.addConstr(y_inj[t, b] + y_abs[t, b] <= 1, name=f'reactive_exclusive_{b}_{t}')

        # Active-power upper bound, limited by C_rate and binary state
        model.addConstr(P_charge[t, b] <= C_rate * cap_pu * y_charge[t, b], name=f'Pcharge_limit_{b}_{t}')
        model.addConstr(P_discharge[t, b] <= C_rate * cap_pu * y_discharge[t, b], name=f'Pdischarge_limit_{b}_{t}')

        # Reactive-power upper bound, limited by inverter oversizing factor
        model.addConstr(Q_inj[t, b] <= (C_rate * INVERTER_OVERSIZE * cap_pu) * y_inj[t, b], name=f'Qinj_limit_{b}_{t}')
        model.addConstr(Q_abs[t, b] <= (C_rate * INVERTER_OVERSIZE * cap_pu) * y_abs[t, b], name=f'Qabs_limit_{b}_{t}')

        # Inverter apparent-power circle constraint (SOCP)
        s_limit_fixed = C_rate * INVERTER_OVERSIZE * cap_pu
        model.addQConstr(
            (P_charge[t, b] + P_discharge[t, b])**2 + (Q_inj[t, b] + Q_abs[t, b])**2 <= s_limit_fixed * s_limit_fixed,
            name=f"BESS_Inverter_Sizing_{b}_{t}"
        )

    # Full-day/full-period energy cycling constraint, with start equal to end
    if num_time_steps > 0:
        t_start = T_dt[0]
        t_end = T_dt[-1]
        model.addConstr(E[t_start, b] == E[t_end, b], name=f'soc_YEAR_CYCLE_{b}')
        # If the daily starting point needs to be fixed at 50%, uncomment the following line and use the constant cap_pu
        # model.addConstr(E[t_start, b] == 0.5 * cap_pu, name=f'soc_START_50_{b}')

model.update()
print("BESS Operational Model Constraints Updated Successfully.")

The current operation model will dispatch BESS only at the following buses: [2058]
BESS Operational Model Constraints Updated Successfully.


In [29]:
model.addConstrs(
    (0 == P_load_dict[(i, t)]
         + quicksum(P_ij[i, j, t] for j in downStream[i])
         - quicksum(P_ij[k, i, t] - r_ij.get((k, i), 0) * l_ij[k, i, t] for k in upStream[i])
         + quicksum(-P_discharge[t, i_c] + P_charge[t, i_c] for i_c in installed_buses if i_c == i)
         #- quicksum(PV_cap[i_p] * pv_profile_map[t] for i_p in candidate_buses if i_p == i)
    for i in commen_num for t in T),
    name='NodePBalance'
)

model.addConstrs(
    (0 == Q_load_dict[(i, t)]
         + quicksum(Q_ij[i, j, t] for j in downStream[i])
         - quicksum(Q_ij[k, i, t] - x_ij.get((k, i), 0) * l_ij[k, i, t] for k in upStream[i])
         + quicksum(-Q_inj[t, i_c] + Q_abs[t, i_c] for i_c in installed_buses if i_c == i)
         #+ quicksum(Q_PV[t, i_q] for i_q in candidate_buses if i_q == i)
     for i in commen_num for t in T),
    name='NodeQBalance'
)

# Active power balance on slack bus
model.addConstrs(
    (0 == -GP_i[i, t] + P_load_dict.get((i, t), 0)
         + quicksum(P_ij[i, j, t] for j in downStream[i])
         - quicksum(P_ij[k, i, t] - r_ij.get((k, i), 0) * l_ij[k, i, t] for k in upStream[i])
     for i in gen['bus'] for t in T),
    name='NodeGPBalance'
)

# Reactive power balance on slack bus
model.addConstrs(
    (0 == -GQ_i[i, t] + Q_load_dict.get((i, t), 0)
         + quicksum(Q_ij[i, j, t] for j in downStream[i])
         - quicksum(Q_ij[k, i, t] - x_ij.get((k, i), 0) * l_ij[k, i, t] for k in upStream[i])
     for i in gen['bus'] for t in T),
    name='NodeGQBalance'
)

# Voltage balance
for _, row in branch.iterrows():
    i, j = int(row['fbus']), int(row['tbus'])
    r_val = r_ij.get((i, j), 0)
    x_val = x_ij.get((i, j), 0)
    for t in T:
        model.addConstr(
            v_i[j, t] == v_i[i, t] - 2 * (r_val * P_ij[i, j, t] + x_val * Q_ij[i, j, t])
                        + (r_val**2 + x_val**2) * l_ij[i, j, t],
            name=f'voltage_{i}_{j}_t{t}'
        )

model.addConstrs(
    (v_i[1, t] == bus1_voltage_mean[t]**2 for t in T),
    name='SlackNode'
)

# Slack bus voltage
#model.addConstrs(
#    (v_i[1, t] == 1.04**2 for t in T),
#    name='SlackNode'
#)

# Second-order cone constraint on line current and power relationships
for t in T:
    for (i, j) in ij:
        constr_name = f'SOC_{i}_{j}_{t}'
        model.addQConstr(
            P_ij[i, j, t] * P_ij[i, j, t] + Q_ij[i, j, t] * Q_ij[i, j, t] 
            <= l_ij[i, j, t] * v_i[i, t],
            name=constr_name
        )

In [30]:
obj = quicksum(
    operation_price[t] * GP_i[i, t]
    for i in gen['bus']
    for t in T
)

model.setObjective(obj, GRB.MINIMIZE)

model.setParam('MIPFocus', 1)
#model.setParam('Cuts', 2)
model.setObjective(obj, GRB.MINIMIZE)

model.setParam('MIPGap', 0.01)
#model.setParam('TimeLimit', 7200)
model.optimize()

Set parameter MIPFocus to value 1
Set parameter MIPGap to value 0.01
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
MIPGap  0.01
MIPFocus  1

Optimize a model with 23232 rows, 23232 columns and 69430 nonzeros
Model fingerprint: 0x8aa2ec3a
Model has 5760 quadratic constraints
Variable types: 23136 continuous, 96 integer (96 binary)
Coefficient statistics:
  Matrix range     [9e-10, 1e+00]
  QMatrix range    [1e+00, 2e+00]
  Objective range  [2e+01, 3e+01]
  Bounds range     [9e-01, 3e+01]
  RHS range        [4e-04, 2e+00]
  QRHS range       [4e-03, 4e-03]
Presolve removed 19247 rows and 17913 columns
Presolve time: 2.39s
Presolved: 3985 rows, 5319 columns, 13976 nonzeros
Presolved model has 1403 quadratic constraint(s)
Variable types: 5281 continuous, 38 integer (38 bina

In [ ]:
import math
import pandas as pd

# Judge optimization status
if model.status == GRB.OPTIMAL:
    print("Optimal Solution Found!\n")
    
    P_ij_all = []
    Q_ij_all = []
    I_ij_all = []      
    V_i_all = []
    GP_i_all = []
    GQ_i_all = []
    P_bess_all = []
    Q_bess_all = []
    total_active_power_all = []
    total_reactive_power_all = []
    total_active_losses_all = []
    total_reactive_losses_all = []
    active_loss_percent_all = []
    reactive_loss_percent_all = []
    
    ref_kV = 6.9 
    I_base_approx = (baseMVA * 1000) / (math.sqrt(3) * ref_kV)

    for t in T:
        # 1 P_ij
        P_ij_data = []
        for (i, j) in ij:
            P_val = P_ij[i, j, t].X * baseMVA * 1000
            P_ij_data.append({'Time': t, 'Branch': (i, j), 'P_ij (kW)': round(P_val, 6)})
        P_ij_df = pd.DataFrame(P_ij_data)
        P_ij_all.append(P_ij_df)
        
        # 2 Q_ij
        Q_ij_data = []
        for (i, j) in ij:
            Q_val = Q_ij[i, j, t].X * baseMVA * 1000
            Q_ij_data.append({'Time': t, 'Branch': (i, j), 'Q_ij (kVar)': round(Q_val, 6)})
        Q_ij_df = pd.DataFrame(Q_ij_data)
        Q_ij_all.append(Q_ij_df)

        # [Added] 1.5 I_ij (Line Current)
        I_ij_data = []
        for (i, j) in ij:
            # l_ij is the squared current (p.u.^2)
            l_val_sq = l_ij[i, j, t].X
            # Prevent numerical errors from causing negative values
            if l_val_sq < 0: l_val_sq = 0
            
            # Calculate current in p.u.
            I_pu = math.sqrt(l_val_sq)
            
            # (Optional) estimate current in amperes for reference only; the accurate value should be based on the specific base voltage of the line
            I_amps = I_pu * baseI 

            I_ij_data.append({
                'Time': t, 
                'Branch': (i, j), 
                'l_ij_sq (p.u.^2)': round(l_val_sq, 6), # squared value
                'Current (p.u.)': round(I_pu, 6),       # per-unit value
                'Current (Est. A)': round(I_amps, 2)    # estimated ampere value
            })
        I_ij_df = pd.DataFrame(I_ij_data)
        I_ij_all.append(I_ij_df)


        # sigma_l_data = []
        # print("\n>>> Checking line overload conditions (Sigma_l)...")
    
        # for t in T:
        #     for (i, j) in ij:
        #         # Get the penalty variable value (p.u.^2)
        #         sigma_val = sigma_l[i, j, t].X
                
        #         # Record only lines with actual overloads, ignoring numerical error
        #         if sigma_val > 1e-6:
        #             # Get the rated limit of this line
        #             limit_amps = line_limit_dict.get((i, j), 228) # default 228 A
        #             limit_sq_pu = (limit_amps / baseI) ** 2
                    
        #             # Calculate actual current and overload amount
        #             # Actual l_ij_sq = limit + sigma
        #             actual_l_sq = limit_sq_pu + sigma_val
                    
        #             actual_current_pu = math.sqrt(actual_l_sq)
        #             limit_current_pu = math.sqrt(limit_sq_pu)
                    
        #             # Convert to amperes
        #             actual_amps = actual_current_pu * baseI
        #             overload_amps = actual_amps - limit_amps
        #             overload_percent = (overload_amps / limit_amps) * 100
                    
        #             sigma_l_data.append({
        #                 'Time': t,
        #                 'Branch': (i, j),
        #                 'Sigma (p.u.^2)': round(sigma_val, 6),
        #                 'Limit (A)': round(limit_amps, 1),
        #                 'Actual (A)': round(actual_amps, 1),
        #                 'Overload (A)': round(overload_amps, 1),
        #                 'Overload (%)': round(overload_percent, 1)
        #             })
    
        # sigma_l_df = pd.DataFrame(sigma_l_data)
    
        # # Print results
        # if not sigma_l_df.empty:
        #     print(f"Found {len(sigma_l_df)} overloaded time-line records:")
        #     # Sort by overload severity in descending order and print the top 20 records
        #     print(sigma_l_df.sort_values(by='Overload (A)', ascending=False).head(20).to_string(index=False))
            
        #     # Save to CSV
        #     sigma_l_df.to_csv('Result_Line_IAS.csv', index=False)
        #     print("Detailed overload report has been saved to 'Result_Line_Overloads.csv'")
        # else:
        #     print("No line overloads were found; all Sigma_l values are 0.")

        
        # 3 V_i
        V_i_data = []
        for i in bus_list:
            # v_i is the squared voltage
            v_val_sq = v_i[i, t].X
            if v_val_sq < 0: v_val_sq = 0
            voltage = math.sqrt(v_val_sq)
            V_i_data.append({'Time': t, 'Bus': i, 'Voltage (p.u.)': round(voltage, 6)})
        V_i_df = pd.DataFrame(V_i_data)
        V_i_all.append(V_i_df)
        
        # 4 GP_i
        GP_i_data = []
        for i in gen['bus']:
            GP_val = GP_i[i, t].X * baseMVA * 1000
            GP_i_data.append({'Time': t, 'Gen_Bus': i, 'GP_i (kW)': round(GP_val, 6)})
        GP_i_df = pd.DataFrame(GP_i_data)
        GP_i_all.append(GP_i_df)
        
        # 5 GQ_i
        GQ_i_data = []
        for i in gen['bus']:
            GQ_val = GQ_i[i, t].X * baseMVA * 1000
            GQ_i_data.append({'Time': t, 'Gen_Bus': i, 'GQ_i (kVar)': round(GQ_val, 6)})
        GQ_i_df = pd.DataFrame(GQ_i_data)
        GQ_i_all.append(GQ_i_df)
        
        # 6 BESS active and reactive power
        P_bess_data = []
        Q_bess_data = []
        for b in candidate_buses:
            
            p_ch = P_charge[t, b].X if (t,b) in P_charge else 0
            p_dis = P_discharge[t, b].X if (t,b) in P_discharge else 0
            q_abs = Q_abs[t, b].X if (t,b) in Q_abs else 0
            q_inj = Q_inj[t, b].X if (t,b) in Q_inj else 0

            P_bess_val = (p_ch - p_dis) * baseMVA * 1000
            Q_bess_val = (q_abs - q_inj) * baseMVA * 1000
            
            P_bess_data.append({'Time': t, 'Bus': b, 'P_bess (kW)': round(P_bess_val, 6)})
            Q_bess_data.append({'Time': t, 'Bus': b, 'Q_bess (kVar)': round(Q_bess_val, 6)})
            
        P_bess_df = pd.DataFrame(P_bess_data)
        Q_bess_df = pd.DataFrame(Q_bess_data)
        P_bess_all.append(P_bess_df)
        Q_bess_all.append(Q_bess_df)
        
        # 7 Total P Q (includes BESS)
        # Sum of slack bus generation + BESS net injection (Charge is load, Discharge is gen)
        # Note: GP_i usually includes Slack bus generation.
        # Net BESS P Injection = (Discharge - Charge). 
        # Usually Total Active Power means Total Load + Losses, or Total Generation.
        # Here assuming Total Generation:
        total_gen_P = GP_i_df['GP_i (kW)'].sum()
        total_bess_P = P_bess_df['P_bess (kW)'].sum() # This is (Ch - Dis), so negative of generation
        # If P_bess is (Charge - Discharge), then Injection is -P_bess
        # Adjust logic based on your definition. Assuming GP_i is the grid input.
        
        total_active_power = total_gen_P  # Simple sum of Generator output
        total_reactive_power = GQ_i_df['GQ_i (kVar)'].sum()
        
        total_active_power_all.append({'Time': t, 'Total Active Gen (kW)': round(total_active_power, 6)})
        total_reactive_power_all.append({'Time': t, 'Total Reactive Gen (kVar)': round(total_reactive_power, 6)})
        
        # 8 Line loss
        # Loss = sum( R * I^2 ) = sum( r_ij * l_ij )
        total_active_losses = sum(r_ij.get((i, j), 0) * l_ij[i, j, t].X for (i, j) in ij) * baseMVA * 1000
        total_reactive_losses = sum(x_ij.get((i, j), 0) * l_ij[i, j, t].X for (i, j) in ij) * baseMVA * 1000
        
        total_active_losses_all.append({'Time': t, 'Total Active Losses (kW)': round(total_active_losses, 6)})
        total_reactive_losses_all.append({'Time': t, 'Total Reactive Losses (kVar)': round(total_reactive_losses, 6)})
        
        # 9 Loss percentage
        active_loss_percent = (total_active_losses / total_active_power) * 100 if total_active_power != 0 else 0
        reactive_loss_percent = (total_reactive_losses / total_reactive_power) * 100 if total_reactive_power != 0 else 0
        active_loss_percent_all.append({'Time': t, 'Active Loss Percent (%)': round(active_loss_percent, 3)})
        reactive_loss_percent_all.append({'Time': t, 'Reactive Loss Percent (%)': round(reactive_loss_percent, 3)})
    
    # Combine all results for every time step
    P_ij_all_df = pd.concat(P_ij_all, ignore_index=True)
    Q_ij_all_df = pd.concat(Q_ij_all, ignore_index=True)
    I_ij_all_df = pd.concat(I_ij_all, ignore_index=True) # [Added]
    V_i_all_df = pd.concat(V_i_all, ignore_index=True)
    GP_i_all_df = pd.concat(GP_i_all, ignore_index=True)
    GQ_i_all_df = pd.concat(GQ_i_all, ignore_index=True)
    P_bess_all_df = pd.concat(P_bess_all, ignore_index=True)
    Q_bess_all_df = pd.concat(Q_bess_all, ignore_index=True)
    
    total_active_power_all_df = pd.DataFrame(total_active_power_all)
    total_reactive_power_all_df = pd.DataFrame(total_reactive_power_all)
    total_active_losses_all_df = pd.DataFrame(total_active_losses_all)
    total_reactive_losses_all_df = pd.DataFrame(total_reactive_losses_all)
    active_loss_percent_all_df = pd.DataFrame(active_loss_percent_all)
    reactive_loss_percent_all_df = pd.DataFrame(reactive_loss_percent_all)
    
    # Combine total results into a single DataFrame
    totals_df = pd.merge(total_active_power_all_df, total_reactive_power_all_df, on='Time')
    totals_df = pd.merge(totals_df, total_active_losses_all_df, on='Time')
    totals_df = pd.merge(totals_df, total_reactive_losses_all_df, on='Time')
    totals_df = pd.merge(totals_df, active_loss_percent_all_df, on='Time')
    totals_df = pd.merge(totals_df, reactive_loss_percent_all_df, on='Time')

    pd.set_option('display.float_format', lambda x: '%.6f' % x)
    
    # Save as csv file
    # P_ij_all_df.to_csv('P_ij_all.csv', index=False)
    # Q_ij_all_df.to_csv('Q_ij_all.csv', index=False)
    I_ij_all_df.to_csv('I_ij_IAS_BESS.csv', index=False) # [Added] save current data
    V_i_all_df.to_csv('V_i_IAS_BESS.csv', index=False)
    # GP_i_all_df.to_csv('GP_i_all.csv', index=False)
    # GQ_i_all_df.to_csv('GQ_i_all.csv', index=False)
    #P_bess_all_df.to_csv('P_bess_IAS.csv', index=False)
    #Q_bess_all_df.to_csv('Q_bess_IAS.csv', index=False)
    #totals_df.to_csv('totals_all.csv', index=False)
    
    print("Successfully saved all results including Line Currents to CSV.")
    
else:
    print("No optimal solution")

In [ ]:
import pandas as pd

sys_baseMVA = baseMVA if 'baseMVA' in locals() or 'baseMVA' in globals() else 10

if model.status == GRB.OPTIMAL:
    print("Optimal Solution Found!\n")
    
    bess_results = []
    
    print(f"{'Bus':<5} | {'Status':<15} | {'Capacity (p.u.)':<15} | {'Capacity (kWh)':<15}")
    print("-" * 60)

    for b in candidate_buses:
        # [Modification 1] Directly obtain capacity from the planning variable E_cap
        # Note: the capacity unit here is usually p.u. energy and needs conversion
        capacity_pu = E_cap[b].X
        
        # [Modification 2] Determine whether BESS is installed based on whether the capacity is significantly greater than 0
        # The planning model usually does not force a binary variable; installation is inferred from whether E_cap is nonzero
        is_installed = 1 if capacity_pu > 1e-4 else 0
        
        # Convert to physical units, assuming the time step is in hours: p.u. power * time -> p.u. energy
        # Capacity (kWh) = Capacity (p.u.) * baseMVA * 1000
        capacity_kwh = capacity_pu * sys_baseMVA * 1000
        
        status_str = "Installed" if is_installed else "Not Installed"
        print(f"{b:<5} | {status_str:<15} | {capacity_pu:<15.6f} | {capacity_kwh:<15.2f}")

        # Output detailed time-series data only when BESS is installed
        if is_installed:
            print(f"   -> Details for Bus {b}:")
            print("   Timestamp            SOC     P_ch(kW)   P_dis(kW)  Q_abs(kVar)  Q_inj(kVar)")
            
            for t in T_dt:
                # SOC calculation
                current_E_pu = E[t, b].X
                # Avoid division by zero
                soc_val = current_E_pu / capacity_pu if capacity_pu > 1e-9 else 0.0
                
                # [Modification 3] Power unit conversion (p.u. -> kW/kVar)
                pch_val  = P_charge[t, b].X * sys_baseMVA * 1000
                pdis_val = P_discharge[t, b].X * sys_baseMVA * 1000
                qabs_val = Q_abs[t, b].X * sys_baseMVA * 1000
                qinj_val = Q_inj[t, b].X * sys_baseMVA * 1000
                
                # Store in a list for generating the DataFrame
                bess_results.append({
                    'Bus': b,
                    'Installed': is_installed,
                    'Capacity_pu': round(capacity_pu, 6),
                    'Capacity_kWh': round(capacity_kwh, 3),
                    'Timestamp': t,
                    'SOC': round(soc_val, 4),
                    'P_charge_kW': round(pch_val, 4),
                    'P_discharge_kW': round(pdis_val, 4),
                    'Q_abs_kVar': round(qabs_val, 4),
                    'Q_inj_kVar': round(qinj_val, 4)
                })
                
                # Print only the first few rows for checking to avoid excessive output
                #if t == T_dt[0] or t == T_dt[1] or t == T_dt[2]:
                print(f"   {t}, {soc_val:.4f}, {pch_val:.2f},     {pdis_val:.2f},      {qabs_val:.2f},       {qinj_val:.2f}")
            print("   ...\n")

    # Save results
    if bess_results:
        bess_df = pd.DataFrame(bess_results)
        bess_df.to_csv('BESS_IAS.csv', index=False)
        print(f"BESS planning results processed. Total data points: {len(bess_df)}")
        # display(bess_df.head())
    else:
        print("Optimization finished, but no BESS capacity was planned (all zero).")

else:
    print("No optimal solution found! Status Code:", model.status)

In [ ]:
# Judge optimization status
if model.status == GRB.OPTIMAL:
    print("Optimal Solution Found!\n")
    print("--- BESS Installation Summary ---")
    print(f"{'Bus ID':<10} | {'Status':<15} | {'Planned Capacity (kWh)'}")
    print("-" * 50)
    
    installed_bess_count = 0
    for b in candidate_buses:
        # In the planning model, directly read the E_cap variable
        capacity = E_cap[b].X*10*1000
        
        # Determine installation: if the capacity is significantly greater than 0, it is considered installed
        is_installed = capacity > 1e-6
        
        if is_installed:
            installed_bess_count += 1
            # Here capacity is a p.u. value; to view kWh, multiply by baseMVA * 1000
            print(f"{b:<10} | {'Installed':<15} | {capacity:.6f}")

    if installed_bess_count == 0:
        print("No BESS installed in the optimal solution.")
    
    print("\n--- End of Summary ---")

else:
    print("No optimal solution found! Status Code:", model.status)